# Independent 5X72 ROOT-order challenge — 2026-10-09

Two single-ligand experiments with the other occupant absent. Original crystal references, prepared starting geometries and captured pose outputs remain separate. Saved observations only; see [workflow](5x72_root_order_workflow.md).

In [1]:
import gzip, json, sys
from devtools.qualify_5x72_root_order import OUTPUT, INPUTS, DATA, sha, comparison_rows
assert sys.version_info[:2] == (3,14)
assert 'molsyssuite@uibcdf_3.14' in sys.executable
content = OUTPUT.read_bytes()
assert sha(content) == '2129d7eed2b73bcf90737b2a6c722e3b8e9269abeba0ca3b1444df51f046634e'
record = json.loads(gzip.decompress(content))
assert record['comparison_rows'] == comparison_rows(record['new_runs'])
assert len(record['new_runs']) == 24
assert sum(r['poses'] for r in record['comparison_rows']) == 99
for name, digest in INPUTS.items():
    assert sha((DATA/name).read_bytes()) == digest
for case in record['cases'].values():
    for control in case['controls'].values():
        assert sha(control['pdbqt'].encode()) == control['sha256']
print('24 new searches, 99 saved poses. No historical 1IEP searches reused.')
print('Scientific base:', record['consumer_base_commit'])
print('Provider source:', record['qualified_provider_commit'])
print('Prepared SDF geometry is not the experimental reference. H29 has no experimental reference.')

24 new searches, 99 saved poses. No historical 1IEP searches reused.
Scientific base: 357cd87f273360a30af4fe67f0a5df7b9b21a4e7
Provider source: 5bd893c85fe8d211663b2b1f865f5f1d2c382a90
Prepared SDF geometry is not the experimental reference. H29 has no experimental reference.


In [2]:
from IPython.display import display, Markdown
rows = record['comparison_rows']
table = ['| Ligand | Order | Effort | Seed | Poses | First heavy RMSD (A) | Closest (A) | First Vina score (kcal/mol) |', '| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |']
for r in rows:
    table.append(f"| {r['ligand']} | {r['order']} | {r['exhaustiveness']} | {r['seed']} | {r['poses']} | {r['first_heavy_rmsd_angstrom']:.4f} | {r['closest_heavy_rmsd_angstrom']:.4f} | {r['first_vina_score_kcal_mol']:.3f} |")
display(Markdown('\n'.join(table)))
for ligand in ('p59','p69'):
    for order in ('native','first_fixed'):
        arm = [r for r in rows if r['ligand']==ligand and r['order']==order]
        print(ligand, order, 'first/set recovered:', sum(r['first_recovered'] for r in arm), sum(r['returned_set_recovered'] for r in arm), '/', len(arm))
print('<=2.5 A positional criterion; no alignment/symmetry correction. Neither order recovers any first pose.')

| Ligand | Order | Effort | Seed | Poses | First heavy RMSD (A) | Closest (A) | First Vina score (kcal/mol) |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| p59 | native | 1 | 7 | 4 | 9.9017 | 5.1459 | -10.979 |
| p59 | native | 1 | 42 | 4 | 9.8824 | 5.2884 | -10.974 |
| p59 | native | 1 | 2026 | 2 | 9.8990 | 9.1370 | -10.970 |
| p59 | native | 8 | 7 | 5 | 9.8777 | 1.1814 | -10.954 |
| p59 | native | 8 | 42 | 5 | 9.9106 | 1.1905 | -10.991 |
| p59 | native | 8 | 2026 | 5 | 9.8956 | 1.1658 | -10.954 |
| p59 | first_fixed | 1 | 7 | 3 | 9.9237 | 9.8478 | -10.981 |
| p59 | first_fixed | 1 | 42 | 4 | 9.8947 | 1.1872 | -11.038 |
| p59 | first_fixed | 1 | 2026 | 3 | 9.8893 | 5.2859 | -10.956 |
| p59 | first_fixed | 8 | 7 | 5 | 9.9112 | 1.2004 | -10.964 |
| p59 | first_fixed | 8 | 42 | 5 | 9.9139 | 1.1872 | -10.980 |
| p59 | first_fixed | 8 | 2026 | 5 | 9.9020 | 1.1839 | -10.987 |
| p69 | native | 1 | 7 | 3 | 5.6485 | 5.5467 | -10.455 |
| p69 | native | 1 | 42 | 4 | 5.6539 | 5.1164 | -10.455 |
| p69 | native | 1 | 2026 | 3 | 16.5991 | 16.5555 | -6.158 |
| p69 | native | 8 | 7 | 5 | 5.6469 | 5.0197 | -10.433 |
| p69 | native | 8 | 42 | 5 | 5.6539 | 5.1164 | -10.481 |
| p69 | native | 8 | 2026 | 5 | 5.6535 | 3.9280 | -10.468 |
| p69 | first_fixed | 1 | 7 | 3 | 5.6530 | 5.1160 | -10.444 |
| p69 | first_fixed | 1 | 42 | 3 | 5.6498 | 4.7035 | -10.413 |
| p69 | first_fixed | 1 | 2026 | 3 | 5.6526 | 5.5509 | -10.455 |
| p69 | first_fixed | 8 | 7 | 5 | 5.6504 | 5.1236 | -10.413 |
| p69 | first_fixed | 8 | 42 | 5 | 5.6499 | 3.8594 | -10.413 |
| p69 | first_fixed | 8 | 2026 | 5 | 5.6527 | 5.1374 | -10.521 |

p59 native first/set recovered: 0 3 / 6
p59 first_fixed first/set recovered: 0 4 / 6
p69 native first/set recovered: 0 0 / 6
p69 first_fixed first/set recovered: 0 0 / 6
<=2.5 A positional criterion; no alignment/symmetry correction. Neither order recovers any first pose.


In [3]:
import base64, itertools
import numpy as np
import pyunitwizard as puw
import dockingmt as dmt
from devtools.qualify_5x72_root_order import measure
cells, count = set(), 0
with puw.context(standard_units=['pm','fs','K','mole','dalton','e','kJ/mol','radians']):
    for run in record['new_runs']:
        case = record['cases'][run['ligand']]
        control = case['controls'][run['order']]
        result = dmt.DockingResult.from_dict(run['result'])
        artifacts = result.provenance['backend_artifacts']
        assert dmt.verify_captured_inputs(artifacts)
        assert artifacts['partner']['sha256'] == control['sha256']
        assert base64.b64decode(artifacts['partner']['content_base64']) == control['pdbqt'].encode()
        assert artifacts['receptor']['sha256'] == INPUTS['5x72_receptor.pdbqt']
        p = result.protocol_info['parameters']
        cells.add((run['ligand'],run['order'],p['seed'],p['exhaustiveness']))
        heavy = control['heavy_pdbqt_indices']
        mapped = control['heavy_pdbqt_to_reference_atom_indices']
        assert sorted(mapped)==list(range(24))
        reference = np.array(case['reference_coordinates_angstrom'])[mapped]
        recalculated = measure(result,case,control)
        for pose, saved, measured in zip(result.poses,run['heavy_atom_metrics'],recalculated,strict=True):
            coordinates = puw.get_value(pose.coordinates,to_unit='angstrom')[heavy]
            independent = np.sqrt(np.mean(np.sum((coordinates-reference)**2,axis=-1)))
            assert abs(saved['heavy_atom_positional_rmsd_angstrom']-independent)<1e-10
            assert abs(measured['heavy_atom_positional_rmsd_angstrom']-independent)<1e-10
            assert saved['recovered']==bool(independent<=2.5)
            count += 1
assert cells == set(itertools.product(('p59','p69'),('native','first_fixed'),(7,42,2026),(1,8)))
assert count == 99
print('All 99 saved poses independently checked against the correct crystal instance under pm/fs; no searches repeated.')
print('Single ligands, vacant other site, explicit preparation and external receptor unassessed. No affinity, stereoselectivity or general ROOT policy claim.')

All 99 saved poses independently checked against the correct crystal instance under pm/fs; no searches repeated.
Single ligands, vacant other site, explicit preparation and external receptor unassessed. No affinity, stereoselectivity or general ROOT policy claim.
